In [1]:
import argparse
import distutils.util
import logging
import os
import sys
import time
from glob import glob

import torch
import torch.distributed as dist
from monai.apps.deepedit.interaction import Interaction

from monai.apps.deepedit.transforms import (
    AddGuidanceSignalDeepEditd,
    AddRandomGuidanceDeepEditd,
    FindDiscrepancyRegionsDeepEditd,
    NormalizeLabelsInDatasetd,
    FindAllValidSlicesMissingLabelsd,
    AddInitialSeedPointMissingLabelsd,
    SplitPredsLabeld,
)
from monai.data import partition_dataset, list_data_collate, Dataset,pad_list_data_collate
from monai.data.dataloader import DataLoader
from monai.data.dataset import PersistentDataset
from monai.engines import SupervisedTrainer
from monai.handlers import (
    CheckpointSaver,
    LrScheduleHandler,
    MeanDice,
    StatsHandler,
    TensorBoardStatsHandler,
    from_engine,
)
from monai.inferers import SimpleInferer
from monai.losses import DiceCELoss
from monai.networks.nets import DynUNet
from monai.transforms import (
    Activationsd,
    AsDiscreted,
    Compose,
    EnsureChannelFirstd,
    LoadImaged,
    Orientationd,
    RandFlipd,
    RandShiftIntensityd,
    RandRotate90d,
    ToNumpyd,
    ToTensord,
    CenterSpatialCropd,
)
from monai.utils import set_determinism
import json
from strategy import random_sampling
import numpy as np
import matplotlib.pyplot as plt
# from ignite.engine import _prepare_batch

In [2]:
def get_network(labels):
    # Network
    network = DynUNet(
        spatial_dims=3,
        in_channels=len(labels) + 1,
        out_channels=len(labels),
        kernel_size=[3, 3, 3, 3, 3, 3],
        strides=[1, 2, 2, 2, 2, [2, 2, 1]],
        upsample_kernel_size=[2, 2, 2, 2, [2, 2, 1]],
        norm_name="instance",
        deep_supervision=False,
        res_block=True,
    )
    return network


def get_pre_transforms(labels, spatial_size):
    t = [
        LoadImaged(keys=("image", "label"), reader="ITKReader"),
        EnsureChannelFirstd(keys=("image", "label")),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        RandFlipd(keys=("image", "label"), spatial_axis=[0], prob=0.10),
        RandFlipd(keys=("image", "label"), spatial_axis=[1], prob=0.10),
        RandFlipd(keys=("image", "label"), spatial_axis=[2], prob=0.10),
        RandRotate90d(keys=("image", "label"), prob=0.10, max_k=3),
        RandShiftIntensityd(keys="image", offsets=0.10, prob=0.50),

        NormalizeLabelsInDatasetd(keys="label", label_names=labels),
        CenterSpatialCropd(keys=["image", "label"],roi_size=spatial_size),
        # Transforms for click simulation
        FindAllValidSlicesMissingLabelsd(keys="label", sids="sids"),	
        AddInitialSeedPointMissingLabelsd(keys="label", guidance="guidance", sids="sids"),
        AddGuidanceSignalDeepEditd(keys="image", guidance="guidance"),
        #
        ToTensord(keys=("image", "label")),
    ]

    return Compose(t)

def get_click_transforms():
    t = [
        Activationsd(keys="pred", softmax=True),
        AsDiscreted(keys="pred", argmax=True),
        ToNumpyd(keys=("image", "label", "pred")),
        # Transforms for click simulation
        FindDiscrepancyRegionsDeepEditd(keys="label", pred="pred", discrepancy="discrepancy"),
        AddRandomGuidanceDeepEditd(
            keys="NA",
            guidance="guidance",
            discrepancy="discrepancy",
            probability="probability",
        ),
        AddGuidanceSignalDeepEditd(keys="image", guidance="guidance"),
        #
        ToTensord(keys=("image", "label")),
    ]

    return Compose(t)


def get_post_transforms(labels):
    t = [
        Activationsd(keys="pred", softmax=True),
        AsDiscreted(
            keys=("pred", "label"),
            argmax=(True, False),
            to_onehot=(len(labels), len(labels)),
        ),
        # This transform is to check dice score per segment/label
        SplitPredsLabeld(keys="pred"),
    ]
    return Compose(t)

In [3]:
prefix = './datasets/RAINE_organ_51'
labels = {
    "left kidney": 1,
    # "right kidney": 2,
    # "pancreas": 3,
    "background": 0,
}
spatial_size = [192, 192, 96]
pre_transforms = get_pre_transforms(labels, spatial_size)

/scratch/user/uqxye5/miniconda/envs/medsam/lib/python3.10/site-packages/monai/transforms/io/array.py:178: UserWarning: required package for reader ITKReader is not installed, or the version doesn't match requirement.
  warnings.warn(


In [13]:
imagelist = glob(os.path.join(prefix, 'trainingset_51', '*.nii.gz'))
labellist = glob(os.path.join(prefix, 'trainingset_51', 'labels', 'final', '*.nii.gz'))
# print('len(imagelist)', len(imagelist), 'len(labellist)', len(labellist))
datalist = [{"image": image_name, "label": label_name} for image_name, label_name in zip(imagelist, labellist)]
# print(datalist[-1])
total_l = len(datalist)
# if len(datalist) <= 3:
#     batch_size = len(datalist)
# else:
#     batch_size = 3

# Check the length of the elements in datalist
# for data in datalist:
#     print(len(data["image"]), len(data["label"]))

# # Check the output of pre_transforms
# for data in datalist:
#     transformed = pre_transforms(data)
#     print(transformed["image"].shape, transformed["label"].shape, '\n label value', np.unique(transformed["label"]))

# print('len(nii_pool)', len(nii_pool), '\tbatch_size', batch_size)
train_ds = Dataset(datalist, pre_transforms)
# print('dataset length', len(train_ds))
# print each data shape in train_ds
# for i, data in enumerate(train_ds):
#     if data["image"].shape != (3, 192, 192, 96) or data["label"].shape != (1, 192, 192, 96):
#         print(i, data['image_meta_dict']['filename_or_obj'])
#         print( data["image"].shape, data["label"].shape)
#         break




from torch.utils.data import DataLoader
# def collate_fn(batch):
#     keys = batch[0].keys()
#     print('keys', keys)
#     print('batch', batch[0]['sids'].shape)
#     return {key: torch.stack([sample[key] for sample in batch]) for key in keys}

from monai.data.utils import pickle_operations
from collections.abc import Mapping
from torch.utils.data._utils.collate import default_collate
from monai.data.meta_obj import MetaObj
from monai.utils import TraceKeys, first
from collections.abc import Sequence

def collate_meta_tensor(batch):
    """collate a sequence of meta tensor sequences/dictionaries into
    a single batched metatensor or a dictionary of batched metatensor"""
    if not isinstance(batch, Sequence):
        raise NotImplementedError()
    elem_0 = first(batch)
    # print('elem_0 keys', elem_0.keys())
    if isinstance(elem_0, MetaObj):
        collated = default_collate(batch)
        meta_dicts = [i.meta or TraceKeys.NONE for i in batch]
        common_ = set.intersection(*[set(d.keys()) for d in meta_dicts if isinstance(d, dict)])
        if common_:
            meta_dicts = [{k: d[k] for k in common_} if isinstance(d, dict) else TraceKeys.NONE for d in meta_dicts]
        collated.meta = default_collate(meta_dicts)
        collated.applied_operations = [i.applied_operations or TraceKeys.NONE for i in batch]
        collated.is_batch = True
        return collated
    if isinstance(elem_0, Mapping):
        return {k: collate_meta_tensor([d[k] for d in batch]) for k in elem_0}
    if isinstance(elem_0, (tuple, list)):
        return [collate_meta_tensor([d[i] for d in batch]) for i in range(len(elem_0))]

    # no more recursive search for MetaTensor
    return default_collate(batch)


def list_collate_fn(batch):
    # print('batch', len(batch)) # 3
    elem = batch[0] # dict dict_keys(['image', 'label', 'image_meta_dict', 'label_meta_dict', 'label_names', 'sids', 'guidance'])
    # print(type(elem))
    # data = [i for k in batch for i in k] if isinstance(elem, list) else batch
    data = batch
    # print(data[0].keys())
    # print(batch==data)
    key = None
    data = pickle_operations(data)  # bc 0.9.0
    # print(data[0].keys())
    # for i in range(len(data)):
    #     print(i)
    #     if 'guidance' not in data[i].keys():
    #         print(elem['image_meta_dict']['filename_or_obj'])
    #         print(elem['image'].shape, elem['label'].shape)

    if isinstance(elem, Mapping):
            # print('Mapping')
            ret = {}
            # print(type(elem))
            print(elem.keys())
            for k in elem:
                key = k
                print('key',k)
                if k == 'sids':
                    print(elem['sids'])
                if 'guidance' not in elem.keys():
                    print('no guidance', elem.keys())
                data_for_batch = [d[key] for d in data]
                # print(len(data_for_batch), data_for_batch[0].shape)
                ret[key] = collate_meta_tensor(data_for_batch)
    # else:
    #     ret = collate_meta_tensor(data)
    return ret


train_loader = DataLoader(train_ds, batch_size=3, shuffle=False, num_workers=0, collate_fn=list_collate_fn)
for i, batch in enumerate(train_loader):
    print('index', i)
    # print(batch.keys())
    # print(batch)
    # print('batch', batch["image"].shape, batch["label"].shape)  # B, C, H, W, D
    

dict_keys(['image', 'label', 'image_meta_dict', 'label_meta_dict', 'label_names', 'sids', 'guidance'])
key image
key label
key image_meta_dict
key label_meta_dict
key label_names
key sids
{'left kidney': [40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73], 'background': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95]}
key guidance
index 0
dict_keys(['image', 'label', 'image_meta_dict', 'label_meta_dict', 'label_names', 'sids', 'guidance'])
key image
key label
key image_meta_dict
key label_meta_dict
key label_names
key sids
{'left kidney': [36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46

IndexError: list index out of range